In [12]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from collections import Counter

In [3]:
# load movie_metadata.csv into a pandas DataFrame
df = pd.read_csv('movie_metadata.csv')

In [7]:
df['plot_keywords'].head(10)

0               avatar|future|marine|native|paraplegic
1    goddess|marriage ceremony|marriage proposal|pi...
2                  bomb|espionage|sequel|spy|terrorist
3    deception|imprisonment|lawlessness|police offi...
4                                                  NaN
5    alien|american civil war|male nipple|mars|prin...
6            sandman|spider man|symbiote|venom|villain
7    17th century|based on fairy tale|disney|flower...
8    artificial intelligence|based on comic book|ca...
9                     blood|book|love|potion|professor
Name: plot_keywords, dtype: str

In [9]:
print(df.head(2))

   color   director_name  num_critic_for_reviews  duration  \
0  Color   James Cameron                   723.0     178.0   
1  Color  Gore Verbinski                   302.0     169.0   

   director_facebook_likes  actor_3_facebook_likes      actor_2_name  \
0                      0.0                   855.0  Joel David Moore   
1                    563.0                  1000.0     Orlando Bloom   

   actor_1_facebook_likes        gross                           genres  ...  \
0                  1000.0  760505847.0  Action|Adventure|Fantasy|Sci-Fi  ...   
1                 40000.0  309404152.0         Action|Adventure|Fantasy  ...   

  num_user_for_reviews language  country  content_rating       budget  \
0               3054.0  English      USA           PG-13  237000000.0   
1               1238.0  English      USA           PG-13  300000000.0   

   title_year actor_2_facebook_likes imdb_score  aspect_ratio  \
0      2009.0                  936.0        7.9          1.78   
1    

In [ ]:
# Extract Top Directors
top_directors = df['director_name'].value_counts().head(20)
print("--- Top 20 Directors ---")
print(top_directors)

# Extract Top Genres (splitting pipe-separated genres)
genres_series = df['genres'].dropna().str.split('|')
all_genres = [genre for sublist in genres_series for genre in sublist]
top_genres = Counter(all_genres).most_common(20)
print("\n--- Top 20 Genres ---")
for genre, count in top_genres:
    print(f"{genre}: {count}")

# Extract Top Actors (across actor_1_name, actor_2_name, actor_3_name)
actor_cols = ['actor_1_name', 'actor_2_name', 'actor_3_name']
all_actors = df[actor_cols].values.flatten()
# Filter out NaN values
all_actors = [actor for actor in all_actors if pd.notna(actor)]
top_actors = Counter(all_actors).most_common(30)

print("\n--- Top 30 Actors ---")
for actor, count in top_actors:
    print(f"{actor}: {count}")

--- Top 20 Directors ---
director_name
Steven Spielberg     26
Woody Allen          22
Martin Scorsese      20
Clint Eastwood       20
Ridley Scott         17
Tim Burton           16
Steven Soderbergh    16
Spike Lee            16
Renny Harlin         15
Oliver Stone         14
Sam Raimi            13
Michael Bay          13
Robert Zemeckis      13
Ron Howard           13
Joel Schumacher      13
Barry Levinson       13
Robert Rodriguez     13
John Carpenter       13
Peter Jackson        12
Shawn Levy           12
Name: count, dtype: int64

--- Top 20 Genres ---
Drama: 2594
Comedy: 1872
Thriller: 1411
Action: 1153
Romance: 1107
Adventure: 923
Crime: 889
Sci-Fi: 616
Fantasy: 610
Horror: 565
Family: 546
Mystery: 500
Biography: 293
Animation: 242
Music: 214
War: 213
History: 207
Sport: 182
Musical: 132
Documentary: 121

--- Top 30 Actors ---
Robert De Niro: 54
Morgan Freeman: 47
Johnny Depp: 41
Bruce Willis: 40
Matt Damon: 38
Steve Buscemi: 37
Liam Neeson: 34
Brad Pitt: 34
Bill Murray: 34


In [ ]:
def extract_features(
    df: pd.DataFrame,
    top_n_genres: int = 20,
    top_n_directors: int = 20,
    top_n_actors: int = 30,
) -> tuple[pd.DataFrame, dict]:
    """Extracts and scales feature vectors for the preference model.
    """
    data = df.copy()


    data["log_gross"] = np.log1p(data["gross"].fillna(data["gross"].median()))
    data["imdb_score_clean"] = data["imdb_score"].fillna(data["imdb_score"].median())
    data["duration_clean"] = data["duration"].fillna(data["duration"].median())

    # 2. Min-Max Scale the Transformed Features
    continuous_cols = ["imdb_score_clean", "log_gross", "duration_clean"]

    scaler = MinMaxScaler()
    scaled_continuous = pd.DataFrame(
        scaler.fit_transform(data[continuous_cols]),
        columns=["num_imdb_score", "num_log_gross", "num_duration"],
        index=data.index,
    )

    # Genre Multi-Hot Encoding (Top N genres)
    genres_series = data["genres"].fillna("").apply(lambda x: [g for g in x.split("|") if g])
    all_genres = [g for sublist in genres_series for g in sublist]
    top_genres = set(pd.Series(all_genres).value_counts().head(top_n_genres).index)

    data["genres_filtered"] = genres_series.apply(
        lambda x: [g for g in x if g in top_genres]
    )

    mlb_genres = MultiLabelBinarizer()
    genres_encoded = pd.DataFrame(
        mlb_genres.fit_transform(data["genres_filtered"]),
        columns=[f"genre_{g}" for g in mlb_genres.classes_],
        index=data.index,
    )

    # Director One-Hot Encoding (Top N directors)
    top_directors = set(data["director_name"].value_counts().head(top_n_directors).index)
    data["director_clean"] = data["director_name"].apply(
        lambda x: x if x in top_directors else "Other"
    )
    directors_encoded = pd.get_dummies(
        data["director_clean"], prefix="director", dtype=int
    )
    if "director_Other" in directors_encoded.columns:
        directors_encoded.drop(columns=["director_Other"], inplace=True)

    # Actor Multi-Hot Encoding (Top N actors across all 3 columns)
    actor_cols = ["actor_1_name", "actor_2_name", "actor_3_name"]
    all_actors = data[actor_cols].values.flatten()
    top_actors = set(
        pd.Series([a for a in all_actors if pd.notna(a)])
        .value_counts()
        .head(top_n_actors)
        .index
    )

    data["actors_combined"] = data[actor_cols].apply(
        lambda row: [a for a in row if a in top_actors], axis=1
    )

    mlb_actors = MultiLabelBinarizer()
    actors_encoded = pd.DataFrame(
        mlb_actors.fit_transform(data["actors_combined"]),
        columns=[f"actor_{a}" for a in mlb_actors.classes_],
        index=data.index,
    )

    # Combine all feature sets
    X = pd.concat(
        [scaled_continuous, genres_encoded, directors_encoded, actors_encoded],
        axis=1,
    )

    metadata = {
        "scaler": scaler,
        "top_genres": list(top_genres),
        "top_directors": list(top_directors),
        "top_actors": list(top_actors),
    }

    return X, metadata


# Example execution:
df = pd.read_csv("movie_metadata.csv")
X, meta = extract_features(df)
print(f"Extracted feature matrix shape: {X.shape}")

Extracted feature matrix shape: (5043, 73)


In [16]:
X

,num_imdb_score,num_gross,num_duration,genre_Action,genre_Adventure,genre_Animation,genre_Biography,genre_Comedy,genre_Crime,genre_Documentary,...,actor_Robert De Niro,actor_Robert Downey Jr.,actor_Robin Williams,actor_Scarlett Johansson,actor_Steve Buscemi,actor_Sylvester Stallone,actor_Tom Cruise,actor_Tom Hanks,actor_Tom Wilkinson,actor_Will Ferrell
0,0.797468,1.000000,0.339286,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0.696203,0.406840,0.321429,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0.658228,0.263080,0.279762,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0.873418,0.589253,0.311508,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0.696203,0.033553,0.190476,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5038,0.772152,0.033553,0.158730,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5039,0.746835,0.033553,0.071429,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
5040,0.594937,0.033553,0.136905,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5041,0.594937,0.000014,0.184524,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
